# 식중독 발생 현황 분석

## 1. 데이터 불러오기

In [1]:
import pandas as pd

### 데이터 읽어오기 + 혹시 모르니까 원본은 냅두고 .copy()로 바꿔놓기

In [2]:
df_raw = pd.read_csv('../data/processed/food_poisoning_2015_2025.csv', encoding="utf-8-sig")

In [3]:
df = df_raw.copy()

## 2. 데이터 검증 단계

In [4]:
df.head()

,연도,월,시설,발생건수,환자수
0,2015,1,가정집,3,8
1,2015,1,기타,4,38
2,2015,1,불명,0,0
3,2015,1,음식점,23,127
4,2015,1,학교,1,12


In [5]:
df.tail()

,연도,월,시설,발생건수,환자수
787,2025,12,기타,2,61
788,2025,12,불명,0,0
789,2025,12,음식점,13,402
790,2025,12,학교,3,161
791,2025,12,학교 외 집단급식,2,56


In [6]:
# 데이터 크기 확인    행: 1월 부터 12월까지 총 11년  시설 종류는 6가지 니까  792행, 열: 연도, 월, 시설, 발생건수, 환자수 5열
df.shape

(792, 5)

In [7]:
# 결측치, dtype 확인
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 792 entries, 0 to 791
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   연도      792 non-null    int64
 1   월       792 non-null    int64
 2   시설      792 non-null    str  
 3   발생건수    792 non-null    int64
 4   환자수     792 non-null    int64
dtypes: int64(4), str(1)
memory usage: 31.1 KB


In [8]:
# 결측치 체크
df.isnull().sum()

연도      0
월       0
시설      0
발생건수    0
환자수     0
dtype: int64

In [14]:
# 중복행 있는지 확인
df.duplicated().sum()

np.int64(0)

In [28]:
# 음수 수치 있는지 확인
(df[['발생건수', '환자수']] < 0).any()

발생건수    False
환자수     False
dtype: bool

In [9]:
# 이상치 확인            # 최대지 3298? 너무 갑자기 크게 나오네? 너 두고보자 일단 의심
df[['발생건수', '환자수']].describe().round(2)

,발생건수,환자수
count,792.00,792.00
mean,4.26,93.13
std,6.19,196.68
min,0.00,0.00
25%,0.00,0.00
50%,2.00,32.00
75%,5.00,119.25
max,37.00,3298.00


In [10]:
df['환자수'].max() / df['환자수'].mean()        # 평균의 35배?...

np.float64(35.41284453422633)

In [11]:
df.loc[df['발생건수'] == 37]      # 일단 37이면 봐줄만은 한가? 일단 패스

,연도,월,시설,발생건수,환자수
117,2016,8,음식점,37,167
183,2017,7,음식점,37,265


In [12]:
df.loc[df['환자수'] == 3298]              # 2018년 9월 사건이 있었나?

,연도,월,시설,발생건수,환자수
265,2018,9,기타,12,3298


### 데이터 검증 중 이상값 조사
2018년 9월 전국 각급 학교 급식으로 제공된 초콜릿 케이크를 섭취한 뒤 살모넬라균(살모넬라 톰슨)에 
감염된 집단 식중독 사고가 크게 발생했다.

-참고자료
https://www.yna.co.kr/view/AKR20180907073551017 , 연합뉴스 기사

### raw_data 와 맞는지 검증
같은 *사이트 연도/시설 별 통계, 연도/월 별 통계 다운받아서 제가 합친 데이터와 일치하는 지 검증

data/raw 폴더에 1. (연도별_월별_식중독_통계_.xls)파일, 2. (연도별_원인시설별_식중독_통계_.xls)파일 이랑 비교

*사이트 = 식품의약품안전처에서 운영하는 식품 안전 나라. readme에 출처 적겠습니다. 비교는 눈으로 했습니다.

In [30]:
df.groupby('연도')[['발생건수', '환자수']].sum() # 연별 총 발생건수, 환자 수 데이터 확인 / 일치

,발생건수,환자수
연도,,
2015,330,5981
2016,399,7162
2017,336,5649
2018,363,11504
2019,286,4075
2020,164,2534
2021,245,5160
2022,311,5501
2023,359,8789


In [31]:
df.groupby('월')[['발생건수', '환자수']].sum() # 월별 총 발생건수, 환자 수 데이터 확인 / 일치

,발생건수,환자수
월,,
1,236,2755
2,148,1895
3,217,4613
4,262,5267
5,307,6517
6,342,7408
7,398,9513
8,405,12740
9,364,11901


In [32]:
df.groupby('시설')[['발생건수','환자수']].sum() # 시설 별 총 발생건수, 환자수 데이터 확인  / 일치

,발생건수,환자수
시설,,
가정집,39,181
기타,515,16453
불명,106,1241
음식점,2002,24940
학교,314,17933
학교 외 집단급식,401,13011


## 3. 데이터 분석

### 3-1연도별 발생 추이

In [37]:
year = df.groupby('연도')[['발생건수', '환자수']].sum()
year

,발생건수,환자수
연도,,
2015,330,5981
2016,399,7162
2017,336,5649
2018,363,11504
2019,286,4075
2020,164,2534
2021,245,5160
2022,311,5501
2023,359,8789


In [43]:
year['환자수 증감률(%)'] = year['환자수'].pct_change() * 100   # diff 으로 처리하려다가 너무 복잡해서 ai썼습니다...
year['건당 평균 환자수'] = round(year['환자수'] / year['발생건수'], 2) # 건당 평균 환자수 추가
year

,발생건수,환자수,건당 평균 환자수,환자수 증감률(%)
연도,,,,
2015,330,5981,18.12,NaN
2016,399,7162,17.95,19.745862
2017,336,5649,16.81,-21.125384
2018,363,11504,31.69,103.646663
2019,286,4075,14.25,-64.577538
2020,164,2534,15.45,-37.815951
2021,245,5160,21.06,103.630624
2022,311,5501,17.69,6.608527
2023,359,8789,24.48,59.770951


### 발생건수는 비슷한데 환자수가 2018년에만 급격하게 발생했던 것을 볼 수 있다. 이것은 2018년도에  전국 학교급식으로 제공된 초콜릿 케이크로 인한 집단 식중독 사고가 영향을 미쳤을 가능성이 있다.

### 3-2 월별 발생 추이

In [47]:
# 월별 데이터
month = df.groupby('월')[['발생건수', '환자수']].sum()

,발생건수,환자수
월,,
1,236,2755
2,148,1895
3,217,4613
4,262,5267
5,307,6517
6,342,7408
7,398,9513
8,405,12740
9,364,11901


In [50]:
# 건당 평균 환자수, 환자수 비중(%) 추가
month['건당 평균 환자수'] = round(month['환자수'] / month['발생건수'], 2)
month['환자수 비중(%)'] = month['환자수'] / month['환자수'].sum() * 100
month

,발생건수,환자수,건당 평균 환자수,환자수 비중(%)
월,,,,
1,236,2755,11.67,3.735137
2,148,1895,12.80,2.569178
3,217,4613,21.26,6.254152
4,262,5267,20.10,7.140823
5,307,6517,21.23,8.835532
6,342,7408,21.66,10.043520
7,398,9513,23.90,12.897409
8,405,12740,31.46,17.272468
9,364,11901,32.70,16.134980


In [75]:
# 계절 별 발생 건수 데이터 생성
season_table = pd.DataFrame({'봄':month.loc[[3,4,5],['발생건수', '환자수']].sum(),
                             '여름':month.loc[[6,7,8],['발생건수', '환자수']].sum(),
                             '가을':month.loc[[9,10,11],['발생건수', '환자수']].sum(),
                             '겨울':month.loc[[12,1,2],['발생건수', '환자수']].sum()}).T

In [76]:
# 계절 별 환자수 비중(%), 건당 평균 환자수 추가
season_table['환자수 비중(%)'] = season_table['환자수'] / season_table['환자수'].sum() * 100
season_table['건당 평균 환자수'] = round(season_table['환자수'] / season_table['발생건수'], 2)
season_table

,발생건수,환자수,환자수 비중(%),건당 평균 환자수
봄,786,16397,22.230507,20.86
여름,1145,29661,40.213398,25.90
가을,848,19674,26.673355,23.20
겨울,598,8027,10.882740,13.42


In [73]:
# 2018년 9 월 식중독 사태 그게 지금 가을 데이터에 어느 정도 영향 줬는지 확인하고 싶다.
autumn_2018_9 = df.loc[(df['연도'] == 2018) & (df['월'] == 9), '환자수'].sum()
a = autumn_2018_9 / season_table.loc['가을', '환자수'] * 100
a.round(2)
# 11년 중 가을 33개월 중에서 1개가 26.63% 차지 한다 영향을 많이 줬다는걸 알 수 있다.

np.float64(26.63)

### 
계절별 발생건수와 환자 수 모두 여름에 가장 높고 겨울에 가장 낮게 나타났다.
여름철은 건당 평균 환자 수 또한 높게 나타나 발생 빈도 뿐 아니라 발생 1건 당 환자 규모에서도 차이를 보였따.
가을철 환자수에는 2018년도 9월 대규모 식중독 환자 발생이 큰 영향을 미친 것으로 볼 수 있다.

### 3-3 시설별 발생 추이

In [86]:
# 시설별 총 발생건수, 환자수
facility = df.groupby('시설')[['발생건수', '환자수']].sum()

# 시설당 환자수 차지하는 비율 추가
facility['환자수 비중(%)'] = facility['환자수'] / facility['환자수'].sum() * 100

# 건당 평균 환자수 추가
facility['건당 평균 환자수'] = (facility['환자수'] / facility['발생건수']).round(2)
facility

,발생건수,환자수,환자수 비중(%),건당 평균 환자수
시설,,,,
가정집,39,181,0.245394,4.64
기타,515,16453,22.306430,31.95
불명,106,1241,1.682507,11.71
음식점,2002,24940,33.812823,12.46
학교,314,17933,24.312965,57.11
학교 외 집단급식,401,13011,17.639881,32.45


In [92]:
# 연도 / 시설별 발생건수, 환자수 데이터
year_facility = df.groupby(['연도', '시설'])[ ['발생건수', '환자수']].sum()

In [90]:
year_facility['발생건수'].unstack()  # 연도별 / 시설별 발생건 수 만 보기

시설,가정집,기타,불명,음식점,학교,학교 외 집단급식
연도,,,,,,
2015,9,54,4,199,38,26
2016,3,73,4,251,36,32
2017,2,52,10,222,27,23
2018,3,67,9,202,44,38
2019,3,48,7,175,24,29
2020,3,15,0,108,13,25
2021,3,44,3,119,21,55
2022,3,43,16,180,25,44
2023,4,54,33,200,21,47


In [93]:
year_facility['환자수'].unstack() # 연도별 / 시설별 환자수 만 보기

시설,가정집,기타,불명,음식점,학교,학교 외 집단급식
연도,,,,,,
2015,34,1641,18,1506,1980,802
2016,16,974,109,2120,3039,904
2017,6,776,294,1994,2153,426
2018,10,4094,66,2323,3136,1875
2019,7,764,61,1409,1214,620
2020,13,280,0,797,401,1043
2021,12,501,7,2705,708,1227
2022,32,1239,108,2117,1102,903
2023,32,2057,225,3526,1027,1922


In [97]:
# 월별 / 시설별 발생건수, 환자수 데이터
month_facility = df.groupby(['월', '시설'])[ ['발생건수', '환자수']].sum()

In [98]:
month_facility['발생건수'].unstack()  # 월별 / 시설별 발생건 수 만 보기

시설,가정집,기타,불명,음식점,학교,학교 외 집단급식
월,,,,,,
1,7,41,7,156,4,21
2,2,29,7,90,3,17
3,1,40,11,106,37,22
4,3,50,5,150,24,30
5,3,55,14,172,29,34
6,4,46,15,183,53,41
7,1,52,12,245,34,54
8,4,40,7,260,47,47
9,3,52,6,222,38,43


In [99]:
month_facility['환자수'].unstack() # 월별 / 시설별 환자수 만 보기

시설,가정집,기타,불명,음식점,학교,학교 외 집단급식
월,,,,,,
1,22,424,77,1510,116,606
2,14,277,52,893,182,477
3,3,689,131,1161,1911,718
4,10,1099,123,1520,1414,1101
5,8,1428,289,2184,1172,1436
6,18,1179,125,2001,2489,1596
7,4,2442,45,3885,1404,1733
8,13,2570,130,3576,4830,1621
9,32,4153,103,3319,2777,1517


### 시설별 분석 결과
학교와 학교 외 집단급식소는 단체로 모이게 되는 만큼 건당 평균 환자수가 상대적으로 높게 나타나고 있다.
환자수 비중 자체는 음식점에서 가장 높게 나타나고 있다.
2018년도에 환자수 데이터 안에 학교는 집단 식중독 사건 때문에 증가했다고 사료되지만 '기타' 데이터가 늘어난 이유는 현재 데이터만으로는 특정 할 수 없다. 추가적인 데이터와 분석이 필요하다.
또한 2020년에는 전체적인 식중독 환자수가 크게 감소하는 양상이 관찰되었는데 이는 코로나19로 인한 환경 및 위생 관리 변화가 영향을 미쳤을 가능성이 있다.